# Train the store person detector on Google Colab (free GPU)

**Before you start**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. In Google Drive, create the folder `MyDrive/retail_videos/` and upload your clips with this layout:
```
retail_videos/store_1/cam1.mp4  cam2.mp4  cam3.mp4  cam5.mp4
retail_videos/store_2/...
```
3. Run the cells top to bottom. Results (trained weights + accuracy report) are saved back to
   `MyDrive/retail_outputs/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## 1. Get the project and install the vision packages

In [ ]:
!git clone -q https://github.com/madhu328/Analytic_Insights_for_Retail_Store.git /content/project
%cd /content/project
!pip -q install -r requirements-pipeline.txt

## 2. Connect Google Drive and find your videos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
VIDEO_DIR = '/content/drive/MyDrive/retail_videos'
OUT_DIR = '/content/drive/MyDrive/retail_outputs'
import os, glob
os.makedirs(OUT_DIR, exist_ok=True)
videos = sorted(glob.glob(f'{VIDEO_DIR}/**/*.mp4', recursive=True))
print(len(videos), 'videos found'); print('\n'.join(videos))
assert videos, f'Upload your clips to {VIDEO_DIR} first'

## 3. Extract frames (1 per second, near-duplicates skipped)
Aim for 1,500+ frames in total. With only a few 2-minute clips, lower `--every` to `0.5`.

In [ ]:
!python -m training.extract_frames "$VIDEO_DIR" --out datasets/raw/images --every 1.0
!ls datasets/raw/images | wc -l

## 4. Pre-label every frame with the large YOLO11x teacher model

In [ ]:
!python -m training.prelabel datasets/raw/images --out datasets/raw/labels --model yolo11x.pt --imgsz 1280

## 5. Review the labels (needed for a trustworthy ≥ 95%)
The teacher's boxes contain mistakes (missed occluded shoppers, mirror reflections, posters).
Training and testing only on unreviewed teacher boxes gives a score that measures agreement with
the teacher, **not** real accuracy.

**Recommended:** run the next cell, download `labels_for_review.zip`, import it into
[CVAT](https://app.cvat.ai) (format *YOLO 1.1*), fix the boxes, export as YOLO 1.1, and upload the
corrected zip to `MyDrive/retail_outputs/labels_reviewed.zip`. Then run the cell after it.

**Quick trial:** skip both cells to train on the unreviewed labels.

In [ ]:
!cd datasets/raw && zip -qr "$OUT_DIR/labels_for_review.zip" images labels && echo "saved to $OUT_DIR/labels_for_review.zip"

In [ ]:
# Use the corrected labels you uploaded (CVAT export: labels are under obj_train_data/)
import zipfile, shutil, pathlib
z = f'{OUT_DIR}/labels_reviewed.zip'
if os.path.exists(z):
    shutil.rmtree('datasets/raw/labels_reviewed', ignore_errors=True)
    zipfile.ZipFile(z).extractall('datasets/raw/labels_reviewed')
    shutil.rmtree('datasets/raw/labels'); pathlib.Path('datasets/raw/labels').mkdir()
    for f in pathlib.Path('datasets/raw/labels_reviewed').rglob('*.txt'):
        if f.name not in ('obj.names', 'train.txt'):
            shutil.copy(f, 'datasets/raw/labels/' + f.name)
    print(len(os.listdir('datasets/raw/labels')), 'reviewed label files in use')
else:
    print('No reviewed labels uploaded: training on the teacher labels')

## 6. Build the dataset: whole videos go to train / val / test (no leakage)

In [ ]:
!python -m training.build_dataset --images datasets/raw/images --labels datasets/raw/labels --out datasets/store_person --val 0.15 --test 0.15

## 7. Fine-tune YOLO11m (about 30-90 min on a T4 depending on frame count)

In [ ]:
!python -m training.train_detector --data datasets/store_person/data.yaml --epochs 120 --imgsz 960 --patience 30

## 8. Save the trained model and report to Google Drive

In [ ]:
!cp models/store_yolo11m.pt reports/detector_metrics.json "$OUT_DIR/"
!cp -r runs/store_yolo11m "$OUT_DIR/training_run" 2>/dev/null || true
import json; r = json.load(open('reports/detector_metrics.json'))
for split in ('val', 'test'):
    m = r.get(split, {})
    if 'precision' in m:
        print(f"{split:5s} precision {m['precision']:.3f}  recall {m['recall']:.3f}  F1 {m['f1']:.3f}  mAP50 {m['map50']:.3f}  (target 0.95)")
    else:
        print(split, m)

## 9. Use the model on your laptop
Download `store_yolo11m.pt` and `detector_metrics.json` from `MyDrive/retail_outputs/`, then put them in the project:
```
Analytic_Insights_for_Retail_Store-main/models/store_yolo11m.pt
Analytic_Insights_for_Retail_Store-main/reports/detector_metrics.json
```
The dashboard's **Model accuracy** tab shows the report, and the pipeline uses the model with
`--model models/store_yolo11m.pt`.